# 01 — ChainPilot Data Exploration

Exploratory analysis of the three client Excel datasets.

**Sources:**
- `Raw_data/Stock management.xlsx` — board material stock ledger
- `Raw_data/GLUE RECORD-1.xlsx` — glue consumable ledger
- `Raw_data/Tree log suppliers Book.xlsx` — supplier delivery records

Run the ETL scripts first to generate catalog and quality reports:
```bash
python -m etl.ingest_catalog
python -m etl.profile_data
```

In [1]:
import os
import json
import re
from collections import defaultdict
from decimal import Decimal, InvalidOperation

import openpyxl
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DATA = os.path.join(ROOT, 'Raw_data')
DATA_DIR = os.path.join(ROOT, 'data')

print('Root:', ROOT)
print('Raw data files:', os.listdir(RAW_DATA))

Root: c:\Users\Amalitech\OneDrive - AmaliTech gGmbH\Desktop\AmaliTechProject\ChainPilot\repo\ChainPilotDE
Raw data files: ['GLUE RECORD-1.xlsx', 'Stock management.xlsx', 'Tree log suppliers Book.xlsx']


## 1. Load Catalog

In [2]:
catalog_path = os.path.join(DATA_DIR, 'catalog', 'catalog.json')
if os.path.exists(catalog_path):
    with open(catalog_path) as f:
        catalog = json.load(f)
    for src in catalog['sources']:
        print(f"\n{src['source_id']} — {src['filename']} ({src.get('file_size_kb', '?')} KB)")
        for sheet in src.get('sheets', []):
            print(f"  [{sheet['sheet_name']}] rows={sheet['total_rows']} cols={sheet['column_count']}")
else:
    print('Catalog not found — run: python -m etl.ingest_catalog')


stock_management — Stock management.xlsx (90.7 KB)
  [6MM] rows=1028 cols=9
  [9MM] rows=1036 cols=11
  [12MM] rows=1000 cols=9
  [18MM] rows=1019 cols=26
  [20MM] rows=1000 cols=9
  [27MM] rows=1000 cols=12
  [33MM] rows=1000 cols=9
  [35MM ] rows=1000 cols=12
  m] rows=1000 cols=12
  [sales form] rows=1064 cols=26

glue_record — GLUE RECORD-1.xlsx (17.6 KB)
  [GLUE A , B, C and D] rows=159 cols=9

tree_log_suppliers — Tree log suppliers Book.xlsx (100.4 KB)
  [Production summary] rows=98 cols=10
  [Received tree logs trucks] rows=248 cols=19
  [MWISENEZA Joseph] rows=19 cols=17
  [NYIRANKUMBUYE Vestine] rows=27 cols=18
  [NYIRANSENGIYUMVA Marceline] rows=59 cols=20
  [UWIMANA Evode] rows=5 cols=17
  [NISENGWE Felicien] rows=14 cols=17
  [NIYONASENZE Alphonse] rows=37 cols=17
  [MUSABYEMARIYA Vestine] rows=7 cols=17
  [MANIRIYO Emmanuel] rows=7 cols=17
  [ESS OIL Ltd] rows=11 cols=17
  [SEBUTOZI Bernard] rows=11 cols=17
  [DUSABYEYEZU Emmanuel] rows=8 cols=17
  [Uwintije Jean D' Amou

## 2. Stock Management — Tab Overview

In [3]:
THICKNESS_RE = re.compile(r'^\d+\s*[Mm]{2}\s*$')

wb_stock = openpyxl.load_workbook(os.path.join(RAW_DATA, 'Stock management.xlsx'), data_only=True)
print(f'All sheets ({len(wb_stock.sheetnames)}): {wb_stock.sheetnames}')

valid_tabs = [s for s in wb_stock.sheetnames if THICKNESS_RE.match(s)]
print(f'\nValid thickness tabs ({len(valid_tabs)}): {valid_tabs}')
print(f'Skipped: {[s for s in wb_stock.sheetnames if not THICKNESS_RE.match(s)]}')

All sheets (10): ['6MM', '9MM', '12MM', '18MM', '20MM', '27MM', '33MM', '35MM ', '14mm', 'sales form']

Valid thickness tabs (9): ['6MM', '9MM', '12MM', '18MM', '20MM', '27MM', '33MM', '35MM ', '14mm']
Skipped: ['sales form']


## 3. Stock Management — Items and Specifications by Tab

In [4]:
rows = []
for tab in valid_tabs:
    ws = wb_stock[tab]
    tab_rows = list(ws.iter_rows(values_only=True))
    # Find header row
    header_idx = next(
        (i for i, r in enumerate(tab_rows)
         if any(str(c).strip().lower() == 'items' for c in r if c)),
        -1
    )
    if header_idx < 0:
        rows.append({'tab': tab, 'items': None, 'specs': [], 'error': 'no header'})
        continue
    
    data = tab_rows[header_idx + 1:]
    first_items = next((str(r[1]).strip() for r in data if len(r) > 1 and r[1]), None)
    all_specs = list({str(r[2]).strip() for r in data if len(r) > 2 and r[2] and not str(r[2]).startswith('=')})
    
    rows.append({
        'tab': tab.strip(),
        'items_value': first_items,
        'display_name': f'{first_items} MDF' if first_items else None,
        'spec_variants': all_specs,
        'variant_count': len(all_specs),
        'canonical_spec': all_specs[0] if all_specs else None,
    })

df = pd.DataFrame(rows)
df

,tab,items_value,display_name,spec_variants,variant_count,canonical_spec
0,6MM,6mm,6mm MDF,"[0.6*122*245, 0.6*122*244, 0.6*122*246]",3,0.6*122*245
1,9MM,9mm,9mm MDF,[0.9*122*244],1,0.9*122*244
2,12MM,12mm,12mm MDF,[1.2*122*244],1,1.2*122*244
3,18MM,18mm,18mm MDF,[1.8*122*244],1,1.8*122*244
4,20MM,20mm,20mm MDF,[0.9*122*244],1,0.9*122*244
5,27MM,27mm,27mm MDF,[2.7*122*244],1,2.7*122*244
6,33MM,33mm,33mm MDF,[3.3*122*244],1,3.3*122*244
7,35MM,33mm,33mm MDF,[3.5*122*244],1,3.5*122*244
8,14mm,14mm,14mm MDF,[1.4*122*244],1,1.4*122*244


## 4. Parse Specification Strings

In [5]:
def parse_spec(raw):
    parts = raw.split('*')
    if len(parts) != 3:
        return None, f'expected 3 segments, got {len(parts)}'
    try:
        return {'thickness': Decimal(parts[0]), 'width': Decimal(parts[1]), 'length': Decimal(parts[2])}, None
    except InvalidOperation as e:
        return None, str(e)

parsed_rows = []
for _, row in df.iterrows():
    if not row['canonical_spec']:
        continue
    dims, err = parse_spec(row['canonical_spec'])
    parsed_rows.append({
        'tab': row['tab'],
        'display_name': row['display_name'],
        'raw_spec': row['canonical_spec'],
        **({} if err else {'thickness': float(dims['thickness']),
                            'width': float(dims['width']),
                            'length': float(dims['length'])}),
        'parse_error': err,
    })

pd.DataFrame(parsed_rows)

,tab,display_name,raw_spec,thickness,width,length,parse_error
0,6MM,6mm MDF,0.6*122*245,0.6,122.0,245.0,None
1,9MM,9mm MDF,0.9*122*244,0.9,122.0,244.0,None
2,12MM,12mm MDF,1.2*122*244,1.2,122.0,244.0,None
3,18MM,18mm MDF,1.8*122*244,1.8,122.0,244.0,None
4,20MM,20mm MDF,0.9*122*244,0.9,122.0,244.0,None
5,27MM,27mm MDF,2.7*122*244,2.7,122.0,244.0,None
6,33MM,33mm MDF,3.3*122*244,3.3,122.0,244.0,None
7,35MM,33mm MDF,3.5*122*244,3.5,122.0,244.0,None
8,14mm,14mm MDF,1.4*122*244,1.4,122.0,244.0,None


## 5. Supplier Workbook — Received Tree Logs Headers

In [6]:
wb_sup = openpyxl.load_workbook(os.path.join(RAW_DATA, 'Tree log suppliers Book.xlsx'), data_only=True)
print('All sheets:', wb_sup.sheetnames[:5], '...')

ws_trucks = wb_sup['Received tree logs trucks']
for i, row in enumerate(ws_trucks.iter_rows(values_only=True)):
    cells = [str(c).strip() for c in row if c is not None and str(c).strip()]
    if len(cells) >= 2:
        print(f'\nHeader row (row {i+1}):')
        for j, h in enumerate(cells):
            print(f'  [{j}] {h!r}')
        break

All sheets: ['Production summary', 'Received tree logs trucks', 'MWISENEZA Joseph', 'NYIRANKUMBUYE Vestine', 'NYIRANSENGIYUMVA Marceline'] ...

Header row (row 4):
  [0] 'Date'
  [1] 'Supplier name'
  [2] 'Plate number'
  [3] 'Log length (m)'
  [4] 'Supplied Wood width (m)'
  [5] 'Supplied Wood length (m)'
  [6] 'Total cubed m3'
  [7] 'Unit price per m3'
  [8] 'Total price'
  [9] 'Unqualified (Rwf)'
  [10] 'Return out'
  [11] 'Balance to be paid'
  [12] 'Paid amount'
  [13] 'Payment date'
  [14] 'Payment Mode'
  [15] 'Cheque number'
  [16] 'Remaining balance'


## 6. GLUE RECORD-1 — Overview

In [7]:
wb_glue = openpyxl.load_workbook(os.path.join(RAW_DATA, 'GLUE RECORD-1.xlsx'), data_only=True)
print('Sheets:', wb_glue.sheetnames)

ws_glue = wb_glue['GLUE A , B, C and D']
glue_rows = list(ws_glue.iter_rows(values_only=True))[:5]
for i, row in enumerate(glue_rows):
    print(f'Row {i+1}: {row}')

Sheets: ['GLUE A , B, C and D']
Row 1: ('INPUT', None, None, None, 'OUTPUT', None, None, None, None)
Row 2: ('Date', 'Items', 'Specifications', 'quantity', 'Date', 'Items', 'Specifications', 'quantity', 'balance')
Row 3: (datetime.datetime(2025, 5, 2, 0, 0), 'glue', '25kg/1bag', 934, datetime.datetime(2025, 4, 5, 0, 0), 'glue', '25kg/1bag', 6, 6011)
Row 4: (datetime.datetime(2025, 7, 10, 0, 0), 'glue', '25kg/1bag', 1120, datetime.datetime(2025, 4, 8, 0, 0), 'glue', '25kg/1bag', 24, 5987)
Row 5: (datetime.datetime(2025, 7, 24, 0, 0), 'glue', '25kg/1bag', 160, datetime.datetime(2025, 4, 9, 0, 0), 'glue', '25kg/1bag', 33, 5954)


## 7. Data Quality Report

In [8]:
report_path = os.path.join(DATA_DIR, 'profiles', 'data_quality_report.json')
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    for src in report['sources']:
        print(f"\n{'='*40}")
        print(f"{src['source_id']} — total_issues={src.get('total_issues', '?')}")
        for sheet in src.get('sheets', []):
            issues = sheet.get('issues', [])
            if issues:
                print(f"  [{sheet['sheet_name']}]")
                for issue in issues:
                    print(f"    >> {issue}")
else:
    print('Quality report not found — run: python -m etl.profile_data')


stock_management — total_issues=143
  [6MM]
    >> Column 'INPUT': mixed types {'str': 1, 'datetime': 77}
    >> Column 'col_1': 64.4% null
    >> Column 'col_2': 64.4% null
    >> Column 'col_3': mixed types {'str': 1, 'int': 31}
    >> Column 'col_3': 63.2% null
    >> Column 'OUTPUT': mixed types {'str': 1, 'datetime': 42}
    >> Column 'OUTPUT': 50.6% null
    >> Column 'col_5': 50.6% null
    >> Column 'col_6': 50.6% null
    >> Column 'col_7': mixed types {'str': 1, 'int': 43}
    >> Column 'col_8': mixed types {'str': 1, 'int': 86}
    >> 7 duplicate data row(s) detected
  [9MM]
    >> Column 'Date': mixed types {'datetime': 84, 'str': 1}
    >> Column 'Items': 55.1% null
    >> Column 'Specifications': 55.1% null
    >> Column 'quantity': 54.1% null
    >> Column 'col_9': 100.0% null
    >> Column 'col_10': 100.0% null
    >> 7 duplicate data row(s) detected
  [12MM]
    >> Column 'INPUT': mixed types {'str': 1, 'datetime': 70}
    >> Column 'col_1': 63.1% null
    >> Column '